# 📗 만들었으면 잰다 — RAG 검색 품질 평가

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

앞 시간까지 우리는 **검색을 만들었습니다.** 문서를 색인하고, 검색을 도구로 감싸고, 에이전트가 필요할 때만 그 도구를 부르게 했지요. 그런데 아직 한 번도 묻지 않은 것이 있습니다 — **그 검색은 얼마나 맞히고 있나요?**

"그럴듯해 보인다" 는 근거가 아닙니다. 이번 시간에는 검색을 **숫자로** 잽니다. 눈금 네 개를 손으로 만들고, 그 숫자를 **어떻게 읽어야 하는지**까지 배웁니다. 만드는 법을 배운 다음에 재는 법을 배우는 이유는 분명합니다 — **재지 못하면 고칠 수도 없기 때문입니다.**

## ⏪ 복습 — 지난 시간까지

| 언제 | 한 일 | 오늘 |
|---|---|---|
| 앞 단원 | 문서를 잘라 색인하고 **RAG 체인**을 만들었다 | 그 색인을 **평가용으로 다시 세운다** |
| 교안 01 | 에이전트의 **사고 루프**(ReAct)를 짚었다 | — |
| 교안 01 | 검색을 **도구로 감싸** 에이전트에 붙였다 | **그 검색이 얼마나 맞히는지 잰다** |
| 교안 01 | 도구 넷을 붙여 라우팅을 확인했다 | 라우팅이 맞아도 **검색이 틀리면 답도 틀린다** |

> 검색을 고르고 고치는 모든 판단 — `k` 를 몇으로 둘지, 조각을 얼마나 크게 자를지 — 은 **잰 숫자 위에서** 이뤄집니다. 오늘이 그 숫자를 만드는 시간입니다.

### 📚 공식 문서 — 오늘 배우는 것들

| 오늘 다루는 것 | 공식 문서 |
|---|---|
| 검색 부품(벡터스토어·리트리버) | [Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval) |
| Chroma 벡터스토어 | [Chroma 연동](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma) |
| HuggingFace 임베딩 | [HuggingFace 임베딩 연동](https://docs.langchain.com/oss/python/integrations/text_embedding/huggingfacehub) |
| 검색기 인자(`search_kwargs`) | [리트리버 레퍼런스](https://reference.langchain.com/python/langchain-core/vectorstores/) |

> 지표 네 개는 LangChain 의 기능이 아니라 **정보검색 분야의 표준 지표**입니다. 오늘은 넷을 **직접 구현해** 식을 확인하고, **그 함수로 평가셋 전체를 잽니다** — 식이 몇 줄로 끝나는 지표라, 무엇을 세고 무엇으로 나누는지를 손으로 확인하는 편이 숫자를 읽는 데 훨씬 낫습니다.

**오늘의 목표**

- [ ] **평가셋** — 질문·정답 라벨·근거 문장이 무엇인지 알고, 라벨을 **조각(청크)** 에 붙이는 이유를 안다.
- [ ] **네 가지 지표** — Hit@K·Precision@K·Recall@K·MRR@K 의 식을 손으로 구현해 확인한다.
- [ ] **전체 평가** — 그 함수로 평가셋 20문항을 재고, 문항별 표와 평균을 함께 읽는다.
- [ ] **읽는 법** — 평균 뒤에 가려진 **실패 문항**을 찾고, Recall 을 **정답 개수와 함께** 읽는다.
- [ ] **K 의 영향** — 같은 검색기를 두 K 로 재서 Precision 과 Recall 의 **트레이드오프**를 확인한다.

아래 준비 셀들을 먼저 실행하세요(임베딩 모델을 내려받느라 처음 한 번은 잠시 걸립니다).

> **이 노트북은 모델을 한 번도 부르지 않습니다.** 검색 평가는 임베딩과 검색만 쓰기 때문입니다 — 그래서 OpenAI 키도 필요 없습니다. **판단의 근거를 수치로 만드는 일에는 모델이 필요 없다**는 것을 오늘 직접 확인하게 됩니다.

---
# 1. 만들었으면 잰다 — 무엇을 가지고 재나

## 왜 재야 할까요?
지금까지 만든 검색이 얼마나 맞히는지 **모른 채로는** `k` 를 몇으로 둘지도, 조각을 얼마나 크게 자를지도 정할 수 없습니다. "좋아 보인다" 는 근거가 아닙니다. 재려면 **문제집**이 필요합니다.

평가셋은 세 가지로 이루어집니다.

| 요소 | 무엇인가 |
|---|---|
| **질문** | 사용자가 실제로 물을 법한 문장 |
| **정답 라벨** | 그 질문의 답이 **실제로 들어 있는 조각의 id** 목록 |
| **근거 문장** | 그 조각 안에서 답이 되는 대목만 **원문 그대로 떼어 온 발췌**(정답 조각 하나에 하나씩) |

<img src="images/평가셋_3요소.png" width="820">

*질문·정답 라벨·근거 문장. 셋이 갖춰져야 잴 수 있습니다.*

> 근거 문장을 함께 적어 두는 이유가 있습니다. 조각의 id 는 **자르는 규칙이 바뀌면 통째로 어긋나지만**, 근거 문장은 어떻게 자르든 원문에 그대로 남습니다. 그래서 색인을 다시 세워도 그 문장을 찾아 **라벨을 다시 붙일 수 있습니다.**

> ⚠️ **라벨은 문서가 아니라 조각(청크)에 붙입니다.** 검색 결과를 문서 단위로 집계하면(같은 문서에서 온 조각을 문서 하나로 합쳐 세면) 답이 없는 조각이 올라와도 "같은 문서니까 맞혔다" 고 세어 버립니다(**허위 크레딧**). 그러면 점수는 높은데 실제 답변 품질은 그대로입니다.

그래서 평가에는 **조각마다 id 가 붙은 색인**이 필요합니다. 교안 01 에서는 안내서 한 건만 색인해 도구로 감쌌으니, 평가용으로는 **두 건 전체**에 라벨을 붙인 자료를 따로 세웁니다 — 개인정보보호위원회가 배포한 **공공 안내서 두 건**(74쪽)과 거기에 라벨을 붙여 둔 **20문항** 평가셋입니다.

그 색인은 **이미 만들어져 함께 배포됩니다** — `data/chroma_day20/` 안의 **`guide_400`** 컬렉션입니다. 안내서 두 건(74쪽)을 `chunk_size=400`·`chunk_overlap=80` 으로 자른 **조각 299개**가 들어 있고, 조각마다 `'{문서id}-{순번}'` 규칙의 id 가 열쇠로 붙어 있습니다(예: `ai33-0`). 평가셋의 정답 라벨이 그 id 로 적혀 있으니 **같은 색인을 써야 견줄 수 있습니다.**

> 매번 다시 색인하지 않는 이유는 둘입니다. 조각 300개를 임베딩하는 동안 수업이 멈추고, **자르는 규칙이 조금이라도 달라지면 조각 id 가 전부 어긋나** 평가셋의 정답 라벨이 무의미해집니다. 그래서 색인은 **한 번 만들어 파일로 고정**하고, 여기서는 열기만 합니다. 만드는 과정은 `부록_평가셋_구축.ipynb` 와 `부록_벡터DB_구축.ipynb` 에 있습니다.

In [ ]:
# [제공 코드] 배포된 평가용 색인을 엽니다 -- 이 셀은 실행만 하세요.
#  같은 임베딩 모델로 열어야 합니다. 다르면 에러 없이 순위만 틀어집니다.
import pandas as pd
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

eval_embeddings = HuggingFaceEmbeddings(model_name='jhgan/ko-sroberta-multitask')   # 임베딩 단원에서 쓴 그 한국어 문장 임베딩 모델
eval_store = Chroma(persist_directory='data/chroma_day20',
                    collection_name='guide_400',
                    embedding_function=eval_embeddings)

# 무엇이 들어 있는지 확인합니다 -- get() 은 저장된 조각을 그대로 꺼내 줍니다.
loaded = eval_store.get()
chunk_ids = loaded['ids']
chunk_texts = loaded['documents']
print(f'조각 {len(chunk_ids)}개 (가장 긴 조각 {max(len(t) for t in chunk_texts)}자)')
print('앞 다섯 개 id:', chunk_ids[:5])

> **가장 긴 조각이 400 를 넘지 않습니다.** 앞 단원에서 손으로 만들었던 문단 청킹은 문단 하나가 상한보다 길면 쪼개지 못해 1,000자가 넘는 덩어리를 남겼습니다. `RecursiveCharacterTextSplitter` 는 문단으로 안 되면 줄로, 줄로 안 되면 문장·낱말로 **내려가며 끝까지 쪼개기 때문에** 약속한 크기를 지킵니다.

In [ ]:
# 평가셋을 읽습니다. gold_chunks 가 '|' 로 이어진 정답 조각 id 목록입니다.
evalset = pd.read_csv('data/guide_eval_chunk.csv')
evalset['정답수'] = evalset['gold_chunks'].str.split('|').apply(len)
print(f'평가셋 {len(evalset)}문항')
display(evalset[['query_id', 'query', 'gold_chunks', '정답수']].head(3))

검색 결과에서 **본문이 아니라 조각 id** 를 꺼내는 함수를 만듭니다. 정답 라벨이 조각 id 이므로 견주려면 같은 것을 꺼내야 합니다.

> `ids=` 로 넘긴 값은 **벡터DB 안에서 그 조각의 열쇠**가 되고, 검색 결과에는 **`Document.id`** 로 실려 돌아옵니다. 그래서 조각 id 를 `metadata` 에 또 적어 둘 필요가 없습니다.

In [ ]:
def search_ids(store, query, k):
    """질문과 가장 가까운 조각 k개의 id 를 순위 순서로 돌려준다."""
    # k 를 그때그때 바꿔 가며 재야 하므로 검색기는 이 자리에서 만든다(색인을 감쌀 뿐이라 비용이 없다).
    retriever = store.as_retriever(search_kwargs={'k': k})
    # invoke 가 돌려주는 것은 Document 이고, 우리가 견줄 것은 본문이 아니라 조각 id 다.
    #  ids= 로 넣어 둔 열쇠가 d.id 로 돌아온다 -- 리스트 순서가 곧 검색 순위다.
    return [d.id for d in retriever.invoke(query)]


first_row = evalset.iloc[0]
print('질문   :', first_row['query'])
print('검색 3 :', search_ids(eval_store, first_row['query'], 3))
print('정답   :', first_row['gold_chunks'].split('|'))

---
# 2. 네 가지 지표 — Hit·Precision·Recall·MRR

검색 결과 상위 K개와 정답 라벨을 놓고 네 가지를 봅니다.

| 지표 | 묻는 것 | 식 |
|---|---|---|
| Hit@K | 상위 K개 **안에 정답이 하나라도** 있나 | 있으면 1, 없으면 0 |
| Precision@K | 상위 K개 중 **몇 개가** 정답인가 | 정답 개수 ÷ K |
| Recall@K | 그 질문의 정답 중 **몇 %를 건졌나** | 찾은 정답 개수 ÷ 전체 정답 개수 |
| MRR@K | 첫 정답이 **몇 위**에 있었나 | 1 ÷ (첫 정답의 순위) |

<img src="images/네가지_지표.png" width="820">

*같은 검색 결과를 네 각도에서 봅니다.*

정답이 3개인 질문에서 상위 3개 안에 정답이 1개 있었고 그 정답이 2위였다고 해 봅시다. Hit@3 = 1, Precision@3 = 1/3, Recall@3 = 1/3, MRR = 1/2 입니다. **맞히기는 했지만 다 건지지는 못했고, 첫 정답도 1위는 아니었다**는 뜻입니다.

먼저 **식을 눈으로 확인**하려고 네 개를 직접 구현해 봅니다. 넷 다 한 줄짜리입니다 — 무엇을 세고 무엇으로 나누는지가 지표의 전부입니다.

In [ ]:
# 네 지표를 직접 구현합니다 -- 이 함수들로 평가셋 전체를 잽니다.
#  predicted -> 검색이 돌려준 조각 id 목록(순위 순), relevant -> 그 질문의 정답 조각 id 목록.
def hit_at_k(predicted, relevant, k):
    """상위 k개 중 정답이 하나라도 있으면 1, 없으면 0."""
    return 1 if any(p in relevant for p in predicted[:k]) else 0


def precision_at_k(predicted, relevant, k):
    """상위 k개 중 정답의 비율(정답 개수 / k)."""
    # 나누는 수는 언제나 k 다 -- 검색 결과가 k 개보다 적게 나와도 k 로 나눈다.
    return sum(1 for p in predicted[:k] if p in relevant) / k


def recall_at_k(predicted, relevant, k):
    """전체 정답 중 상위 k개가 건진 비율(찾은 정답 수 / 전체 정답 수)."""
    # 세는 것은 Precision 과 같고 나누는 수만 다르다 -- 여기는 그 질문의 정답 개수.
    #  그래서 정답 개수가 k 보다 많으면 이 값은 1.0 이 될 수가 없다.
    return sum(1 for p in predicted[:k] if p in relevant) / len(relevant)


def mrr_at_k(predicted, relevant, k):
    """첫 정답 순위의 역수(1위면 1, 2위면 0.5 ...). 상위 k개 안에 없으면 0."""
    # enumerate 의 두 번째 인자 1 은 순위를 0 이 아니라 1 부터 세게 한다.
    for rank, p in enumerate(predicted[:k], 1):
        if p in relevant:
            # 처음 만난 정답에서 바로 끝낸다 -- MRR 이 보는 것은 '첫' 정답의 순위뿐이다.
            return 1 / rank
    return 0.0

In [ ]:
# 손으로 계산한 값과 코드가 같은지 먼저 맞춰 봅니다.
#  검색 결과 3개 중 2위 하나만 정답이고, 그 질문의 정답은 모두 3개인 상황입니다.
predicted_example = ['a-0', 'b-1', 'c-2']   # 검색 결과 상위 3개
relevant_example = ['b-1', 'd-0', 'e-1']    # 정답 3개

print('Hit@3      ', hit_at_k(predicted_example, relevant_example, 3), '(정답 b-1 이 들어 있으므로 1)')
print('Precision@3', round(precision_at_k(predicted_example, relevant_example, 3), 3), '(3개 중 1개 = 0.333)')
print('Recall@3   ', round(recall_at_k(predicted_example, relevant_example, 3), 3), '(정답 3개 중 1개 = 0.333)')
print('MRR@3      ', mrr_at_k(predicted_example, relevant_example, 3), '(첫 정답이 2위 = 0.5)')

## 이제 이 함수로 평가셋 전체를 잽니다

네 값이 손으로 계산한 것과 같습니다. 식이 맞는 것을 확인했으니, **같은 함수로 20문항을 전부** 재면 됩니다.

재는 함수는 **색인과 K 를 인자로** 받게 만듭니다 — 뒤에서 K 만 바꿔 다시 잴 것이기 때문입니다. 문항마다 검색은 **한 번만** 하고, 그 결과 하나로 네 지표를 모두 계산합니다.

In [ ]:
# 라벨을 '문항 id -> 정답 조각 목록' 사전으로 만들어 둡니다.
#  gold_chunks 는 CSV 한 칸에 담느라 '|' 로 이어 붙인 문자열이라, 견주려면 목록으로 되돌려야 합니다.
gold_map = {row.query_id: row.gold_chunks.split('|') for row in evalset.itertuples()}


def evaluate(store, k):
    """문항마다 네 지표를 재서 DataFrame 으로 돌려준다."""
    rows = []
    for row in evalset.itertuples():
        # 검색은 문항마다 한 번만 합니다 -- 지표가 네 개라고 네 번 검색할 이유가 없습니다.
        predicted = search_ids(store, row.query, k)
        relevant = gold_map[row.query_id]
        # 네 지표 모두 위에서 만든 함수로 냅니다 -- 검색 결과와 정답 목록만 있으면 됩니다.
        rows.append({'query_id': row.query_id,
                     'Hit': hit_at_k(predicted, relevant, k),
                     'P': precision_at_k(predicted, relevant, k),
                     'R': recall_at_k(predicted, relevant, k),
                     'MRR': mrr_at_k(predicted, relevant, k)})
    return pd.DataFrame(rows)


scores = evaluate(eval_store, 3)
print('잰 문항 수:', len(scores))

In [ ]:
# 평균보다 '질문별 표' 를 먼저 봅니다 -- 어느 질문이 틀렸는지가 평균보다 먼저 알아야 할 것입니다.
detail = scores.merge(evalset[['query_id', 'query', '정답수']], on='query_id')
display(detail[['query_id', 'Hit', 'P', 'R', 'MRR', '정답수', 'query']].round(3))

In [ ]:
# 그다음이 평균입니다.
print('K = 3 평균')
print(scores[['Hit', 'P', 'R', 'MRR']].mean().round(3).to_string())

---
# 3. 읽는 법 — 평균 뒤에 가려진 것

**1. 평균은 실패를 뭉갠다.** 먼저 물을 것은 "평균이 몇이냐" 가 아니라 **"어느 질문이 틀렸느냐"** 입니다. 위 표에서 `Hit` 이 0 인 줄을 먼저 보세요.

In [ ]:
# Hit 이 0 인 문항을 꺼내 실제로 읽어 봅니다.
#  Hit 이 0 이면 상위 3개 안에 정답이 '하나도' 없었다는 뜻입니다 -- 이 문항들이 고칠 거리입니다.
failed = detail[detail['Hit'] == 0]
print('상위 3개에서 정답을 못 찾은 문항:', failed['query_id'].tolist())
print()
for row in failed.itertuples():
    print(f'[{row.query_id}] {row.query}')
    print('  정답:', gold_map[row.query_id])
    print('  검색:', search_ids(eval_store, row.query, 3))
    print()

평균 하나만 보고 있었다면 이 문항들은 보이지 않았을 것입니다. 실패를 읽으면 **무엇을 고쳐야 할지**가 보입니다.

**2. Recall 이 낮은 것은 검색 탓이 아니라 질문 탓일 수 있습니다.** 정답 개수는 **질문의 넓이**를 따라갑니다. 이 평가셋에서 정답이 하나뿐인 문항을 보면 "프롬프트에 적힌 주민등록번호를 어떻게 걸러내나" 처럼 **묻는 것이 하나**입니다. 반대로 정답이 대여섯 개인 문항은 "공격자 입장에서 점검할 **항목들**", "목적 외로 쓸 수 있는 **경우들**" 같은 **열거형 질문**입니다 — 답이 안내서 여러 쪽에 흩어져 있으니 라벨도 여러 개가 됩니다.

그러면 산수가 따라옵니다. 정답이 6개인 질문을 K=3 으로 재면 Recall 은 아무리 잘해야 3/6 = 0.5 입니다. **K 보다 정답 개수가 많으면 Recall 은 1.0 이 될 수가 없습니다.** 그래서 Recall 은 항상 정답 개수와 함께 읽습니다.

> **실무에서 이 신호를 어떻게 쓰나요.** 평가셋을 만들 때 어떤 문항의 정답 라벨이 자꾸 불어나면, 먼저 **질문이 여러 질문을 하나로 묶은 것은 아닌지** 의심하세요. "점검 항목을 알려 줘" 는 사실 여러 질문이라 쪼개는 편이 낫습니다. 쪼갤 수 없는 성격이라면 — 실제로 그런 질문은 들어옵니다 — 그 문항의 낮은 Recall 을 검색기 탓으로 돌리지 말고 **답변에 넣는 조각 수(K)가 그 질문에는 애초에 모자란다**고 읽어야 합니다. 열거형 질문에 K=3 이면, 검색이 완벽해도 답변은 반쪽입니다.

In [ ]:
# 정답이 하나뿐인 문항만 모으면 Recall 과 Hit 이 같아지는지 확인합니다.
#  정답이 1개면 Recall 의 나누는 수도 1 이라, 맞히면 1.0 못 맞히면 0.0 -- Hit 과 같은 모양이 됩니다.
single = evalset[evalset['정답수'] == 1]['query_id']
only_single = scores[scores['query_id'].isin(single)]   # 그 문항들만 골라 낸 점수표

print(f"단일 정답 {len(only_single)}문항 -- Hit {only_single['Hit'].mean():.3f} / "
      f"Recall {only_single['R'].mean():.3f}")
print(f"전체 {len(scores)}문항        -- Hit {scores['Hit'].mean():.3f} / "
      f"Recall {scores['R'].mean():.3f}")

단일 정답 문항만 보면 Hit 과 Recall 이 **똑같습니다.** 전체로 보면 갈라집니다 — 네 지표가 서로 다른 것을 말한다는 증거입니다.

---
# 4. K 를 바꾸면 무엇이 달라지나 — 두 값을 나란히 재기

`k` 는 검색 결과 중 **상위 몇 개를 채택할지** 정하는 파라미터입니다. k 를 바꿔도 색인·임베딩·순위 계산은 그대로이고, **어디까지 잘라 쓸지**만 달라집니다. 두 값으로 재서 나란히 놓아 봅니다.

In [ ]:
# 같은 색인, 같은 평가셋. 바뀐 것은 K 하나뿐입니다.
compare = pd.DataFrame([{'K': k, **evaluate(eval_store, k)[['Hit', 'P', 'R', 'MRR']].mean().round(3)}
                        for k in (3, 10)])
display(compare)

K 가 커지면 Hit 과 Recall 은 오르고 Precision 은 떨어집니다. 조각을 더 많이 보여 주니 그 안에 정답이 들어올 확률은 높아지지만, 함께 딸려 오는 관련 없는 조각의 비율도 늘어납니다. 이것이 **트레이드오프**입니다.

> **"K 를 키우면 점수가 오른다" 는 것은 검색이 좋아졌다는 뜻이 아닙니다** — 같은 결과를 더 넓은 창으로 봤을 뿐입니다. **K 는 쓰임새에 맞춰 정합니다.** 답변 생성에 조각 3개를 넣을 거라면 K=3 이 맞는 눈금입니다.

### 🖐️ 함께 따라하기 — 지표를 질문 출처별로 나눠 보기

평가셋에는 질문을 어디서 얻었는지가 `질문출처` 열에 적혀 있습니다. 출처별로 나눠 봅니다.

1. `scores` 와 `evalset[['query_id', '질문출처', '정답수']]` 를 `query_id` 로 합치세요.
2. `질문출처` 로 묶어 `Hit`·`R`·`MRR` 의 평균을 내세요. **`정답수` 의 평균도 함께** 내세요.
3. 두 출처에서 `Hit` 과 `R` 이 어느 쪽으로 갈리는지 보세요.

**확인 기준**: 한쪽이 모든 지표에서 높은 게 아닙니다. 그 차이를 **검색기의 실력 차이로 읽으면 안 되는** 이유는 방금 배운 "Recall 은 정답 개수와 함께 읽는다" 가 그대로 답입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) scores 와 evalset[['query_id', '질문출처', '정답수']] 를 query_id 로 합친다
# 2) 질문출처 로 묶어 Hit·R·MRR 과 정답수 의 평균을 낸다
# 3) 표를 출력하고 Hit 과 R 이 어느 쪽으로 갈리는지 본다

### ✅ 바로 확인 퀴즈

**1.** 정답 라벨을 조각이 아니라 **문서**에 붙이면 무엇이 잘못되나요?

<details><summary>정답 보기</summary>

답이 실제로 없는 조각이 올라와도 **같은 문서라는 이유로 맞혔다고 세게** 됩니다(허위 크레딧). 점수는 높아지는데 실제 답변 품질은 그대로입니다.

</details>

**2.** 어떤 열거형 문항의 정답 조각이 **5개**인데 **K=2** 로 쟀습니다. Recall 의 최댓값은? 그리고 그 숫자를 보고 무엇을 해야 하나요?

<details><summary>정답 보기</summary>

**2/5 = 0.4** 입니다. 상위 2개밖에 못 보므로 정답 5개 중 최대 2개만 건질 수 있습니다. 이때 손댈 곳은 **검색기가 아니라 질문이나 K** 입니다 — 질문을 쪼개거나, 그 유형에는 K 를 키우거나. Recall 은 **정답 개수와 함께** 읽어야 이 판단이 나옵니다.

</details>

**3.** K 를 3에서 10으로 키웠더니 Hit 이 올랐습니다. 검색이 좋아진 걸까요?

<details><summary>정답 보기</summary>

아닙니다. **같은 검색 결과를 더 넓은 창으로 봤을 뿐**입니다. 검색기는 한 글자도 바뀌지 않았습니다. Precision 은 오히려 떨어집니다 — 이것이 트레이드오프입니다.

</details>

**4.** Precision@K 는 왜 검색 결과 개수가 아니라 **K** 로 나누나요?

<details><summary>정답 보기</summary>

**K 개를 보여 주기로 하고 잰 값이기 때문**입니다. 검색이 3개를 요청받아 2개만 돌려줬다고 2로 나누면, 적게 돌려줄수록 점수가 좋아지는 이상한 지표가 됩니다. 분모를 K 로 고정해야 **같은 K 끼리 비교**할 수 있습니다.

</details>

---
## 이번 강의 정리

| 개념 | 핵심 | 코드 |
|---|---|---|
| 평가셋 | 질문 + **조각 단위** 정답 라벨 + 근거 문장 | `guide_eval_chunk.csv` |
| 조각 단위 라벨 | 문서 단위로 집계하면 **허위 크레딧**이 생긴다 | 조각 id `'{문서id}-{순번}'` |
| 색인은 고정 | 자르는 규칙이 바뀌면 조각 id 가 어긋나 라벨이 무의미해진다 | 배포된 `chroma_day20` 의 `guide_400` 컬렉션을 **열어서** 쓴다 |
| Hit@K | 상위 K개 안에 정답이 **하나라도** 있나 | `hit_at_k` |
| Precision@K | **꺼내 온 것 중** 몇 개가 정답인가 | `precision_at_k` (분모는 K) |
| Recall@K | **찾았어야 할 것 중** 몇 개를 건졌나 | `recall_at_k` (분모는 정답 개수) |
| MRR@K | 첫 정답이 **몇 위**에 있었나 | `mrr_at_k` |
| 평균보다 실패 문항 | 평균은 실패를 뭉갠다 | `detail[detail['Hit'] == 0]` |
| Recall 읽는 법 | 정답 개수가 K 보다 많으면 1.0 이 불가능 | `정답수` 열과 함께 |
| K 의 영향 | 검색 순위는 그대로, **채택 개수**만 달라진다 | `evaluate(store, k)` |

- **재지 못하면 고칠 수도 없습니다.** 오늘 만든 네 숫자가 앞으로 바꾸는 모든 것의 기준선입니다.
- **숫자 하나로 판단하지 않습니다.** 네 지표는 서로 다른 것을 말하고, 평균은 실패를 가립니다.
- **식을 알면 숫자를 읽을 수 있습니다.** 네 지표는 정보검색의 표준 정의를 그대로 옮긴 것이라, 여기서 만든 함수는 다른 검색기·다른 자료에도 그대로 씁니다.
- 평가셋을 **어떻게 만드는지**가 궁금하다면 `부록_평가셋_구축.ipynb` 를 보세요(참고 자료).

## ⏭️ 예고 — 다음 단원: 에이전트 데이터 분석 자동화

지금까지는 도구를 **우리가 골라 붙여** 두고 모델이 그중에서 고르게 했습니다. 다음 단원에서는 에이전트가 **여러 단계로 이뤄진 일**(데이터를 모으고, 분석하고, 그림으로 그리는)을 스스로 계획하고 실행하게 만듭니다. 그때도 오늘의 태도는 그대로입니다 — **만들었으면 잽니다.**

수고하셨습니다!